In [1]:
import sys
import pickle

# TO CHANGE
BASEDIR = "../../"
sys.path.insert(0, BASEDIR)

In [2]:
from src.graph_main import RemoteKnowledgeGraph, RemoteKnowledgeGraphConfig
from src.knowledge_graph_model import GraphModelConfig, EmbeddingsModelConfig
from src.db_drivers.graph_driver import GraphDriverConfig, GraphDBConnectionConfig, DEFAULT_INMEMORYGRAPH_CONFIG
from src.db_drivers.vector_driver import VectorDriverConfig, EmbedderModelConfig, VectorDBConnectionConfig
from src.db_drivers.kv_driver import KeyValueDriverConfig, KVDBConnectionConfig, DEFAULT_INMEMORYKV_CONFIG

from src.qa_pipeline import QAPipelineConfig
from src.qa_pipeline.query_parser import QueryLLMParserConfig
from src.qa_pipeline.knowledge_comparator import KnowledgeComparatorConfig

from src.qa_pipeline.knowledge_retriever import KnowledgeRetrieverConfig
from src.qa_pipeline.knowledge_retriever.AStarTripletsRetriever import AStarGraphSearchConfig
from src.qa_pipeline.knowledge_retriever.BFSTripletsRetriever import BFSSearchConfig
from src.qa_pipeline.knowledge_retriever.MixturedTripletsRetriever import MixturedGraphSearchConfig

from src.qa_pipeline.answer_generator import QALLMGeneratorConfig

from src.memorize_pipeline import MemPipelineConfig, LLMExtractorConfig, LLMUpdatorConfig

from src.utils import Logger, ReaderMetrics
from src.utils.data_structs import TripletCreator

c:\Users\nikit\anaconda3\envs\LLM\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


#### 1. Загружем датасет с триплетами, на основе которого будет построен граф знаний

In [3]:
PKL_GRAPH_PATH = 'C:/Users/nikit/temp_files/pickled_graphs/all_triplets.pkl'

with open(PKL_GRAPH_PATH, 'rb') as f:
    formated_triplets = pickle.load(f)

In [4]:
print(len(formated_triplets))

5141


#### 2. Задаём конфигурацию графа знаний

In [5]:
# in-memory storage
GRAPH_STORAGE_CONFIG = GraphDriverConfig(db_vendor='inmemory_graph', db_config=DEFAULT_INMEMORYGRAPH_CONFIG)
KV_STORAGE_CONFIG = KeyValueDriverConfig(db_vendor='inmemory_kv', db_config=DEFAULT_INMEMORYKV_CONFIG)

In [ ]:
#
LANGUAGE = 'auto' # 'ru' , 'en', 'auto

#
RETRIEVER_NAME = 'mixture' # 'astar', 'bfs', 'mixture'
RETRIEVER_HYPERP = MixturedGraphSearchConfig() # AStarGraphSearchConfig, BFSSearchConfig, MixturedGraphSearchConfig

# embedder hyperp
DEVICE = 'cuda'
EMBEDDER_MODEL_PATH = '../../models/intfloat/multilingual-e5-small'

# vector dbs hyperp
NODES_DB_PATH = '../../data/graph_structures/vectorized_nodes/testing'
TRIPLETS_DB_PATH = '../../data/graph_structures/vectorized_triplets/testing'
NEED_TO_CLEAR = True

In [ ]:
inmemory_kg_config = RemoteKnowledgeGraphConfig(
    graph_struct_config=GraphModelConfig(driver_config=GRAPH_STORAGE_CONFIG),
    embedds_struct_config=EmbeddingsModelConfig(
        nodesdb_driver_config=VectorDriverConfig(db_config=VectorDBConnectionConfig(
            path=NODES_DB_PATH, db_name='vectorized_nodes', need_to_clear=NEED_TO_CLEAR)),
        tripletsdb_driver_config=VectorDriverConfig(db_config=VectorDBConnectionConfig(
            path=TRIPLETS_DB_PATH, db_name='vectorized_triplets', need_to_clear=NEED_TO_CLEAR)),
        embedder_config=EmbedderModelConfig(model_name_or_path=EMBEDDER_MODEL_PATH, device=DEVICE)),
    qa_pipeline_config=QAPipelineConfig(
        query_parser_config=QueryLLMParserConfig(lang=LANGUAGE),
        knowledge_comparator_config=KnowledgeComparatorConfig(),
        knowledge_retriever_config=KnowledgeRetrieverConfig(
            retriever_method=RETRIEVER_NAME,retriever_config=RETRIEVER_HYPERP,
            cache_config=KV_STORAGE_CONFIG),
        answer_generator_config=QALLMGeneratorConfig(lang=LANGUAGE)),
    mem_pipeline_config=MemPipelineConfig(
        extractor_config=LLMExtractorConfig(lang=LANGUAGE),
        updator_config=LLMUpdatorConfig(lang=LANGUAGE)),
    log=Logger('log/main'))

#### 3. Инициализируем граф знаний

In [6]:
rkg_main = RemoteKnowledgeGraph(config=inmemory_kg_config)

c:\Users\nikit\anaconda3\envs\LLM\Lib\site-packages\huggingface_hub\file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


#### 4. Добавляем в граф загруженные триплеты

In [9]:
rkg_main.kg_model.graph_struct.create_triplets(formated_triplets)
formated_triplets_wid = [triplet for triplet in rkg_main.kg_model.graph_struct.db_conn.triplets_ids.values()]
rkg_main.kg_model.embeddings_struct.add_triplets(formated_triplets_wid)

100%|██████████| 41/41 [00:17<00:00,  2.41it/s]


#### 5. Q&A

In [10]:
examples_questions = [
    "Какой ущерб государству от потери одного гражданина?", # 8.2 mln
    "Сколько экстренных вызовов было принято в 2022 году?", # >10 mln
    "Сколько экстренных вызовов было передано в экстренные службы в 2022 году?", # 78 675
    "На чем основан успех Яндекса?", # на сильной команде  и технологиях собственной разработки
    "Перечисли 8 умных устройств умного дома, продаваемых сбером в 2023 году" # Управляющий хаб  • Сценарная кнопка  • Розетка  • Детский светильник  • Датчики открытия  • Датчики движения  • Датчики протечки  • Датчики температуры и влажности
]

In [11]:
for question in examples_questions:
    print(rkg_main.answer_question(question))
    print("=" * 35)

Средняя оценка ущерба для государства от смерти или тяжелых травм одного человека составляет 8,2 млн рублей (данные ВШЭ за 2015 год).
В 2022 году было принято и обработано более 10 миллионов экстренных вызовов, из которых 78 675 были переданы в экстренные службы.
В 2022 году было передано в экстренные службы 78 675 экстренных вызовов.
Успех Яндекса основан на сильной команде и технологиях собственной разработки. Практически все продукты и сервисы компании созданы на основе этих технологий. Виртуальный ассистент Алиса, помимо умных колонок и ТВ-станций, присутствует в браузере, картах, навигаторе, приложениях Яндекс и Дом с Алисой, а также позволяет интегрировать устройства компании и сторонних производителей в систему умного дома. Компания успешно развивает множество бизнес-моделей, включая рекламный бизнес, райдтех, электронную коммерцию, фудтех, видео- и аудиостриминг, доставку-логистику, облачные технологии и многое другое.
Умные устройства, продаваемые Сбербанком в 2023 году, включ